In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [6]:
DATA_DIR = Path("data/raw")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test_users.csv"

# 1. Подготовка данных к работе

## Загрузка данных

In [21]:
train = pd.read_parquet(TRAIN_PATH)

## Анализ данных

In [22]:
train.shape

(43000347, 7)

In [24]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 43000347 entries, 0 to 43000346
Data columns (total 7 columns):
 #   Column                Dtype   
---  ------                -----   
 0   uid                   uint32  
 1   timestamp             int64   
 2   item_id               uint32  
 3   is_organic            uint8   
 4   played_ratio_pct      float64 
 5   track_length_seconds  float64 
 6   event_type            category
dtypes: category(1), float64(2), int64(1), uint32(2), uint8(1)
memory usage: 1.4 GB


In [23]:
train.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,1,23850979,358453,0,96.0,145.0,listen
1,1,23851159,688547,0,100.0,180.0,listen
2,1,23851564,19131,0,100.0,245.0,listen
3,1,23851719,798591,0,100.0,160.0,listen
4,1,23851889,557996,0,100.0,170.0,listen


In [25]:
train.describe()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
count,4.300035e+07,4.300035e+07,4.300035e+07,4.300035e+07,4.182048e+07,4.182048e+07
mean,4.991880e+03,3.708203e+07,4.710566e+05,5.206316e-01,6.462953e+01,2.073706e+02
std,2.887943e+03,6.929920e+06,2.686429e+05,4.995742e-01,4.413305e+01,8.415808e+01
min,1.000000e+00,2.381809e+07,1.000000e+00,0.000000e+00,0.000000e+00,5.000000e+00
25%,2.479000e+03,3.128874e+07,2.389250e+05,0.000000e+00,7.000000e+00,1.650000e+02
50%,5.026000e+03,3.758673e+07,4.700480e+05,1.000000e+00,1.000000e+02,2.000000e+02
75%,7.451000e+03,4.322883e+07,7.051110e+05,1.000000e+00,1.000000e+02,2.350000e+02
max,1.000000e+04,4.794476e+07,9.340570e+05,1.000000e+00,1.590000e+02,2.495000e+03


In [26]:
train.isna().sum()

uid                           0
timestamp                     0
item_id                       0
is_organic                    0
played_ratio_pct        1179866
track_length_seconds    1179866
event_type                    0
dtype: int64

In [28]:
train["event_type"].value_counts()

event_type
listen       41820481
like           781435
unlike         281924
dislike         97411
undislike       19096
Name: count, dtype: int64

In [30]:
print("Уникальных пользователей:", train["uid"].nunique())
print("Уникальных треков:", train["item_id"].nunique())

Уникальных пользователей: 9844
Уникальных треков: 881514


Пользователей сильно меньше, чем треков -> матрица взаимодействий ожидаемо сильно разреженная

In [31]:
train_users = set(train["uid"].unique())
test_user_ids = set(test_users["user_id"])

known_users = train_users & test_user_ids
new_users = test_user_ids - train_users

print("Всего тестовых пользователей:", len(test_user_ids))
print("Пользователей с историей:", len(known_users))
print("Пользователей без истории:", len(new_users))

Всего тестовых пользователей: 9213
Пользователей с историей: 9062
Пользователей без истории: 151


151 тестовых пользователей нет в train - согласуется с тз

# 2. Разбиение данных на train/val

## Граница

По тз: 
"Сплит по времени: граница проходит по 90% квантилю времени прослушиваний. Train содержит все типы событий до неё."

Найдем этот 90% квантилю времени прослушиваний:

In [32]:
listen_mask = train["event_type"].eq("listen")
time_cutoff = train.loc[listen_mask, "timestamp"].quantile(0.9)

In [33]:
print("Временная граница:", time_cutoff)

Временная граница: 46146954.0


## Делим

In [34]:
train_split = train[train["timestamp"] <= time_cutoff].copy()
val_split = train.loc[
    (train["timestamp"] > time_cutoff) &
    (train["event_type"] == "listen"),
    ["uid", "item_id"]
].copy()

train_split содержит все типы событий до временной границы

val_split содержит только целевые прослушивания после границы

# 3. Popularity Baseline

В качестве бейзлайна возьмем рекомендацию всем пользователям самых поппулярных треков

Количество прослушиваний треков:

In [35]:
track_popularity = (
    train_split[train_split["event_type"] == "listen"]
    .groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

Топ 10 треков:

In [36]:
top10_tracks = track_popularity.head(10).index.tolist()

In [37]:
top10_tracks

[894940, 337739, 433438, 776946, 543001, 642739, 535546, 221507, 58794, 472495]

Целевые прослушивания:

In [40]:
valid_relevant = (
    val_split.groupby("uid")["item_id"]
    .apply(set)
    .to_dict()
)

In [41]:
def ndcg_at_k(recommendations, relevant_items, k=10):
    relevant_items = set(relevant_items)

    if not relevant_items:
        return 0.0

    dcg = sum(
        1 / np.log2(rank + 1)
        for rank, item in enumerate(recommendations[:k], start=1)
        if item in relevant_items
    )

    ideal_hits = min(len(relevant_items), k)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg

In [42]:
baseline_scores = [
    ndcg_at_k(top10_tracks, relevant_items, k=10)
    for relevant_items in valid_relevant.values()
]

In [43]:
baseline_ndcg = np.mean(baseline_scores)
print(f"Popularity Baseline NDCG@10: {baseline_ndcg:.6f}")

Popularity Baseline NDCG@10: 0.150227


Отправная точка - NDCG@10 = 0.150227

# 4. Кандидатогенерация

Метрики: Recall@K, HitRate@K  

In [47]:
def recall_at_k(recommendations, relevant_items, k=100):
    relevant_items = set(relevant_items)

    if not relevant_items:
        return 0.0

    recommended = set(recommendations[:k])
    return len(recommended & relevant_items) / len(relevant_items)

In [49]:
def hit_rate_at_k(recommendations, relevant_items, k=100):
    relevant_items = set(relevant_items)

    if not relevant_items:
        return 0.0

    return int(bool(set(recommendations[:k]) & relevant_items))

## iALS

In [55]:
from scipy.sparse import csr_matrix

In [56]:
def build_interaction_matrix(
    df,
    user_col="uid",
    item_col="item_id",
    event_col="event_type"
):
    # количество прослушиваний каждой пары пользователь–трек
    interactions = (
        df[df[event_col] == "listen"]
        .groupby([user_col, item_col])
        .size()
        .reset_index(name="count")
    )

    user_idx, user_ids = pd.factorize(interactions[user_col])
    item_idx, item_ids = pd.factorize(interactions[item_col])

    R = csr_matrix(
        (
            interactions["count"].to_numpy(dtype=np.float32),
            (user_idx, item_idx)
        ),
        shape=(len(user_ids), len(item_ids))
    )

    return R, user_ids, item_ids

In [57]:
R, user_ids, item_ids = build_interaction_matrix(train_split)

print("Размер матрицы:", R.shape)
print("Количество ненулевых элементов:", R.nnz)
print("Плотность:", R.nnz / (R.shape[0] * R.shape[1]))

Размер матрицы: (8711, 781188)
Количество ненулевых элементов: 10877202
Плотность: 0.001598429980779925


In [59]:
from implicit.als import AlternatingLeastSquares

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [62]:
def fit_ials(
    R,
    factors=64,
    regularization=0.01,
    iterations=20,
    alpha=20,
    num_threads=4
):
    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        alpha=alpha,
        num_threads=num_threads,
        random_state=2501
    )

    model.fit(R)

    return model

In [63]:
ials_model = fit_ials(
    R,
    factors=64,
    regularization=0.01,
    iterations=20,
    alpha=20
)

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:21<00:00,  4.09s/it]


In [64]:
def generate_ials_candidates(
    model,
    R,
    user_idx,
    n_candidates=300
):
    item_indices, scores = model.recommend(
        userid=user_idx,
        user_items=R[user_idx],
        N=n_candidates,
        filter_already_liked_items=False
    )

    return item_indices, scores

In [65]:
def generate_ials_recommendations(
    model,
    R,
    user_ids,
    user_to_idx,
    n_candidates=300
):
    recommendations = {}

    for uid in user_ids:
        if uid not in user_to_idx:
            recommendations[uid] = []
            continue

        user_idx = user_to_idx[uid]
        item_indices, scores = generate_ials_candidates(
            model,
            R,
            user_idx,
            n_candidates=n_candidates
        )
        recommendations[uid] = [
            (item_ids[item_idx], float(score))
            for item_idx, score in zip(item_indices, scores)
        ]

    return recommendations

In [66]:
ials_recommendations = generate_ials_recommendations(
    model=ials_model,
    R=R,
    user_ids=list(valid_relevant.keys()),
    user_to_idx={uid: idx for idx, uid in enumerate(user_ids)},
    n_candidates=300
)

In [67]:
def evaluate_recommendations(
    recommendations,
    valid_relevant,
    candidate_sizes=(100, 300)
):
    results = []

    for k in candidate_sizes:
        recalls = []
        hit_rates = []

        for uid, relevant_items in valid_relevant.items():
            candidates = [
                item for item, score in recommendations.get(uid, [])
            ]

            recalls.append(
                recall_at_k(candidates, relevant_items, k)
            )

            hit_rates.append(
                hit_rate_at_k(candidates, relevant_items, k)
            )

        results.append({
            "method": "iALS",
            "K": k,
            "Recall@K": np.mean(recalls),
            "HitRate@K": np.mean(hit_rates)
        })

    return pd.DataFrame(results)

In [68]:
ials_results = evaluate_recommendations(
    ials_recommendations,
    valid_relevant
)

ials_results

,method,K,Recall@K,HitRate@K
0,iALS,100,0.069474,0.861081
1,iALS,300,0.163635,0.907672


In [69]:
alpha_values = [10, 20, 40]

ials_tuning_results = []

for alpha in alpha_values:
    print(f"Training iALS with alpha={alpha}")

    model = fit_ials(
        R,
        factors=64,
        regularization=0.01,
        iterations=20,
        alpha=alpha
    )

    recommendations = generate_ials_recommendations(
        model=model,
        R=R,
        user_ids=list(valid_relevant.keys()),
        user_to_idx={uid: idx for idx, uid in enumerate(user_ids)},
        n_candidates=300
    )

    results = evaluate_recommendations(
        recommendations,
        valid_relevant
    )

    results["alpha"] = alpha
    ials_tuning_results.append(results)

ials_tuning_results = pd.concat(
    ials_tuning_results,
    ignore_index=True
)

Training iALS with alpha=10


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:21<00:00,  4.09s/it]


Training iALS with alpha=20


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:22<00:00,  4.10s/it]


Training iALS with alpha=40


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:22<00:00,  4.11s/it]


In [70]:
ials_tuning_results

,method,K,Recall@K,HitRate@K,alpha
0,iALS,100,0.077239,0.874131,10
1,iALS,300,0.176157,0.909013,10
2,iALS,100,0.069474,0.861081,20
3,iALS,300,0.163635,0.907672,20
4,iALS,100,0.059846,0.833028,40
5,iALS,300,0.145695,0.901573,40


In [71]:
best_alpha = 10

In [72]:
factors_values = [32, 64, 128]

ials_factors_results = []

for factors in factors_values:
    print(f"Training iALS with factors={factors}")

    model = fit_ials(
        R,
        factors=factors,
        regularization=0.01,
        iterations=20,
        alpha=best_alpha
    )
    recommendations = generate_ials_recommendations(
        model=model,
        R=R,
        user_ids=list(valid_relevant.keys()),
        user_to_idx={uid: idx for idx, uid in enumerate(user_ids)},
        n_candidates=300
    )
    results = evaluate_recommendations(
        recommendations,
        valid_relevant
    )

    results["factors"] = factors
    ials_factors_results.append(results)

Training iALS with factors=32


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:23<00:00,  1.20s/it]


Training iALS with factors=64


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:21<00:00,  4.10s/it]


Training iALS with factors=128


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [02:54<00:00,  8.72s/it]


In [73]:
ials_factors_results = pd.concat(
    ials_factors_results,
    ignore_index=True
)
ials_factors_results

,method,K,Recall@K,HitRate@K,factors
0,iALS,100,0.073569,0.875473,32
1,iALS,300,0.161587,0.906574,32
2,iALS,100,0.077239,0.874131,64
3,iALS,300,0.176157,0.909013,64
4,iALS,100,0.082526,0.878766,128
5,iALS,300,0.194702,0.912794,128


In [74]:
best_factor = 128

In [78]:
import joblib

joblib.dump(ials_model, "models/ials_model.pkl")

['models/ials_model.pkl']

In [79]:
ials_best_config = {
    "factors": 128,
    "alpha": 10,
    "regularization": 0.01,
    "iterations": 20
}

In [81]:
ials_recommendations_best = generate_ials_recommendations(
    model=model,
    R=R,
    user_ids=list(valid_relevant.keys()),
    user_to_idx={uid: idx for idx, uid in enumerate(user_ids)},
    n_candidates=300
)

In [82]:
ials_ndcg_scores = []

for uid, relevant_items in valid_relevant.items():
    recommendations = [
        item for item, score in ials_recommendations_best.get(uid, [])
    ]

    ials_ndcg_scores.append(
        ndcg_at_k(recommendations, relevant_items, k=10)
    )

ials_ndcg = np.mean(ials_ndcg_scores)

print(f"iALS NDCG@10: {ials_ndcg:.6f}")

iALS NDCG@10: 0.158706


## BPR

In [83]:
from implicit.bpr import BayesianPersonalizedRanking

In [84]:
R_binary = R.copy()
R_binary.data[:] = 1

In [85]:
bpr_model = BayesianPersonalizedRanking(
    factors=128,
    learning_rate=0.05,
    regularization=0.01,
    iterations=30,
    num_threads=4,
    random_state=42
)

bpr_model.fit(R_binary)

100%|█████████████████████████████████████████████████| 30/30 [06:07<00:00, 12.24s/it, train_auc=95.16%, skipped=7.64%]


In [86]:
def generate_bpr_recommendations(
    model,
    R,
    user_ids,
    item_ids,
    n_candidates=300
):
    user_to_idx = {
        uid: idx for idx, uid in enumerate(user_ids)
    }

    known_users = [
        uid for uid in valid_relevant
        if uid in user_to_idx
    ]

    user_indices = np.array(
        [user_to_idx[uid] for uid in known_users]
    )

    user_items = R[user_indices]

    item_indices, scores = model.recommend(
        userid=user_indices,
        user_items=user_items,
        N=n_candidates,
        filter_already_liked_items=False
    )

    recommendations = {}

    for row, uid in enumerate(known_users):
        recommendations[uid] = [
            (item_ids[item_idx], float(score))
            for item_idx, score in zip(
                item_indices[row],
                scores[row]
            )
        ]

    # Пользователи без истории в train
    for uid in valid_relevant:
        if uid not in recommendations:
            recommendations[uid] = []

    return recommendations

In [87]:
bpr_recommendations = generate_bpr_recommendations(
    model=bpr_model,
    R=R_binary,
    user_ids=user_ids,
    item_ids=item_ids,
    n_candidates=300
)

In [88]:
bpr_results = evaluate_recommendations(
    bpr_recommendations,
    valid_relevant
)

display(bpr_results)

,method,K,Recall@K,HitRate@K
0,iALS,100,0.065344,0.859373
1,iALS,300,0.139011,0.896817


In [89]:
bpr_ndcg_scores = []

for uid, relevant_items in valid_relevant.items():
    recommendations = [
        item for item, score
        in bpr_recommendations.get(uid, [])
    ]

    bpr_ndcg_scores.append(
        ndcg_at_k(recommendations, relevant_items, k=10)
    )

bpr_ndcg = np.mean(bpr_ndcg_scores)

print(f"BPR NDCG@10: {bpr_ndcg:.6f}")

BPR NDCG@10: 0.206755


In [90]:
def merge_recommendations(
    rec1,
    rec2,
    n_candidates=1000
):
    merged = {}

    all_users = set(rec1) | set(rec2)

    for uid in all_users:
        scores = {}

        for item, score in rec1.get(uid, []):
            scores.setdefault(item, {})["ials"] = score

        for item, score in rec2.get(uid, []):
            scores.setdefault(item, {})["bpr"] = score

        candidates = []

        for item, model_scores in scores.items():
            candidates.append({
                "item_id": item,
                "ials_score": model_scores.get("ials", 0.0),
                "bpr_score": model_scores.get("bpr", 0.0)
            })

        candidates.sort(
            key=lambda x: (
                x["ials_score"] > 0,
                x["bpr_score"] > 0
            ),
            reverse=True
        )

        merged[uid] = candidates[:n_candidates]

    return merged

In [91]:
merged_candidates = merge_recommendations(
    ials_recommendations_best,
    bpr_recommendations,
    n_candidates=600
)

merged_recommendations = {
    uid: [
        (x["item_id"], x["ials_score"] + x["bpr_score"])
        for x in candidates
    ]
    for uid, candidates in merged_candidates.items()
}

In [92]:
def evaluate_union_recall(
    rec1,
    rec2,
    valid_relevant
):
    recalls = []
    hit_rates = []

    for uid, relevant_items in valid_relevant.items():
        items1 = {
            item for item, score in rec1.get(uid, [])
        }
        items2 = {
            item for item, score in rec2.get(uid, [])
        }

        # Объединение кандидатов без дубликатов
        candidates = items1 | items2

        relevant_items = set(relevant_items)

        recalls.append(
            len(candidates & relevant_items) / len(relevant_items)
            if relevant_items else 0.0
        )

        hit_rates.append(
            int(bool(candidates & relevant_items))
        )

    return {
        "method": "iALS + BPR union",
        "Recall": np.mean(recalls),
        "HitRate": np.mean(hit_rates)
    }

In [93]:
union_results = pd.DataFrame([
    evaluate_union_recall(
        ials_recommendations_best,
        bpr_recommendations,
        valid_relevant
    )
])

union_results

,method,Recall,HitRate
0,iALS + BPR union,0.242965,0.920722


In [96]:
from catboost import CatBoostRanker, Pool

In [97]:
from collections import defaultdict
import numpy as np
import pandas as pd

# Популярность треков по количеству прослушиваний
item_popularity = (
    train_split[train_split["event_type"] == "listen"]
    .groupby("item_id")
    .size()
)

# Размер истории пользователя
user_history_size = pd.Series(
    np.diff(R.indptr),
    index=user_ids
).to_dict()


def build_ranker_dataset(
    ials_recs,
    bpr_recs,
    valid_relevant,
    top_n=100
):
    rows = []

    for uid, relevant_items in valid_relevant.items():
        relevant_items = set(relevant_items)

        ials = {
            item: (score, rank)
            for rank, (item, score)
            in enumerate(ials_recs.get(uid, [])[:top_n], start=1)
        }

        bpr = {
            item: (score, rank)
            for rank, (item, score)
            in enumerate(bpr_recs.get(uid, [])[:top_n], start=1)
        }

        candidate_items = set(ials) | set(bpr)

        history_size = user_history_size.get(uid, 0)

        for item in candidate_items:
            ials_score, ials_rank = ials.get(item, (0.0, top_n + 1))
            bpr_score, bpr_rank = bpr.get(item, (0.0, top_n + 1))

            rows.append({
                "uid": uid,
                "item_id": item,
                "ials_score": ials_score,
                "bpr_score": bpr_score,
                "ials_rank": ials_rank,
                "bpr_rank": bpr_rank,
                "in_ials": int(item in ials),
                "in_bpr": int(item in bpr),
                "item_popularity": np.log1p(
                    item_popularity.get(item, 0)
                ),
                "user_history_size": np.log1p(history_size),
                "target": int(item in relevant_items)
            })

    return pd.DataFrame(rows)


ranker_data = build_ranker_dataset(
    ials_recommendations_best,
    bpr_recommendations,
    valid_relevant,
    top_n=100
)

print("Размер таблицы:", ranker_data.shape)
print("Количество позитивных примеров:", ranker_data["target"].sum())
print("Количество пользователей:", ranker_data["uid"].nunique())

Размер таблицы: (1353551, 11)
Количество позитивных примеров: 192816
Количество пользователей: 7824


In [98]:
display(ranker_data.head())

print(
    ranker_data.groupby("uid")["target"]
    .sum()
    .describe()
)

print(
    "Пользователей хотя бы с одним позитивным кандидатом:",
    ranker_data.groupby("uid")["target"].sum().gt(0).sum()
)

,uid,item_id,ials_score,bpr_score,ials_rank,bpr_rank,in_ials,in_bpr,item_popularity,user_history_size,target
0,1,885260,0.986710,4.960860,11,22,1,1,9.359880,5.442418,0
1,1,810508,0.895523,5.131737,93,13,1,1,8.815222,5.442418,0
2,1,926224,0.895954,4.765963,92,40,1,1,9.087834,5.442418,0
3,1,181267,0.943439,0.000000,43,101,1,0,9.485317,5.442418,0
4,1,675859,0.000000,4.824088,101,32,0,1,6.536692,5.442418,0


count    7824.000000
mean       24.644172
std        21.710450
min         0.000000
25%         8.000000
50%        19.000000
75%        36.000000
max       137.000000
Name: target, dtype: float64
Пользователей хотя бы с одним позитивным кандидатом: 7377


In [99]:
from sklearn.model_selection import train_test_split

user_targets = (
    ranker_data.groupby("uid")["target"]
    .sum()
)

users_with_positive = user_targets[user_targets > 0].index
users_without_positive = user_targets[user_targets == 0].index

train_users, test_users_ranker = train_test_split(
    users_with_positive,
    test_size=0.2,
    random_state=42
)

print("Пользователей для обучения:", len(train_users))
print("Пользователей для проверки:", len(test_users_ranker))
print("Пользователей без позитивов:", len(users_without_positive))

Пользователей для обучения: 5901
Пользователей для проверки: 1476
Пользователей без позитивов: 447


In [100]:
from catboost import CatBoostRanker, Pool

FEATURES = [
    "ials_score",
    "bpr_score",
    "ials_rank",
    "bpr_rank",
    "in_ials",
    "in_bpr",
    "item_popularity",
    "user_history_size"
]

train_data = ranker_data[
    ranker_data["uid"].isin(train_users)
].copy()

test_data = ranker_data[
    ranker_data["uid"].isin(test_users_ranker)
].copy()

# Группы одного пользователя должны идти подряд
train_data = train_data.sort_values("uid").reset_index(drop=True)
test_data = test_data.sort_values("uid").reset_index(drop=True)

train_pool = Pool(
    data=train_data[FEATURES],
    label=train_data["target"],
    group_id=train_data["uid"]
)

test_pool = Pool(
    data=test_data[FEATURES],
    label=test_data["target"],
    group_id=test_data["uid"]
)

print("Train:", train_data.shape)
print("Test:", test_data.shape)

Train: (1019647, 11)
Test: (255459, 11)


In [101]:
ranker = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=600,
    learning_rate=0.08,
    depth=8,
    random_seed=42,
    thread_count=4,
    verbose=100,
    early_stopping_rounds=50
)

ranker.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)

0:	test: 0.1509433	best: 0.1509433 (0)	total: 832ms	remaining: 8m 18s
100:	test: 0.3285933	best: 0.3294937 (95)	total: 1m 8s	remaining: 5m 38s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.3294936775
bestIteration = 95

Shrink model to first 96 iterations.


CatBoostRanker(depth=8, early_stopping_rounds=50, eval_metric='NDCG:top=10', iterations=600, learning_rate=0.08, loss_function='YetiRank', random_seed=42, thread_count=4, verbose=100)

In [102]:
# Предсказания CatBoost на контрольной выборке
test_data = test_data.copy()
test_data["catboost_score"] = ranker.predict(test_pool)


def evaluate_ranker_order(data, score_col, ascending=False):
    scores = []

    for uid, group in data.groupby("uid", sort=False):
        group = group.sort_values(
            score_col,
            ascending=ascending
        )

        recommendations = group["item_id"].tolist()
        relevant_items = set(
            group.loc[group["target"] == 1, "item_id"]
        )

        scores.append(
            ndcg_at_k(
                recommendations,
                relevant_items,
                k=10
            )
        )

    return np.mean(scores)


comparison = pd.DataFrame([
    {
        "method": "iALS",
        "NDCG@10": evaluate_ranker_order(
            test_data, "ials_rank", ascending=True
        )
    },
    {
        "method": "BPR",
        "NDCG@10": evaluate_ranker_order(
            test_data, "bpr_rank", ascending=True
        )
    },
    {
        "method": "CatBoost YetiRank",
        "NDCG@10": evaluate_ranker_order(
            test_data, "catboost_score", ascending=False
        )
    }
])

display(comparison.sort_values("NDCG@10", ascending=False))

,method,NDCG@10
2,CatBoost YetiRank,0.329512
1,BPR,0.238142
0,iALS,0.186168


In [103]:
def generate_history_candidates(
    train_split,
    user_ids,
    cutoff,
    n_7d=300,
    n_30d=150,
    n_all=100
):
    listens = train_split.loc[
        train_split["event_type"].eq("listen"),
        ["uid", "item_id", "timestamp"]
    ]

    # Количество прослушиваний за всё время
    all_counts = (
        listens.groupby(["uid", "item_id"])
        .size()
        .rename("cnt_all")
        .reset_index()
    )

    # Прослушивания за последние 7 дней
    recent_7d = listens.loc[
        listens["timestamp"] >= cutoff - 7 * 86400
    ]

    counts_7d = (
        recent_7d.groupby(["uid", "item_id"])
        .size()
        .rename("cnt_7d")
        .reset_index()
    )

    # Прослушивания за последние 30 дней
    recent_30d = listens.loc[
        listens["timestamp"] >= cutoff - 30 * 86400
    ]

    counts_30d = (
        recent_30d.groupby(["uid", "item_id"])
        .size()
        .rename("cnt_30d")
        .reset_index()
    )

    history = (
        all_counts
        .merge(counts_7d, on=["uid", "item_id"], how="left")
        .merge(counts_30d, on=["uid", "item_id"], how="left")
        .fillna({"cnt_7d": 0, "cnt_30d": 0})
    )

    # Ранги внутри каждого пользователя
    history["rank_7d"] = (
        history.groupby("uid")["cnt_7d"]
        .rank(method="first", ascending=False)
    )

    history["rank_30d"] = (
        history.groupby("uid")["cnt_30d"]
        .rank(method="first", ascending=False)
    )

    history["rank_all"] = (
        history.groupby("uid")["cnt_all"]
        .rank(method="first", ascending=False)
    )

    # Объединяем три источника исторических кандидатов
    history = history[
        (history["rank_7d"] <= n_7d)
        | (history["rank_30d"] <= n_30d)
        | (history["rank_all"] <= n_all)
    ].copy()

    # Сохраняем score и ранг для дальнейшего ранжирования
    history["hist_score"] = (
        np.log1p(history["cnt_7d"])
        + np.log1p(history["cnt_30d"])
        + np.log1p(history["cnt_all"])
    )

    history = history.sort_values(
        ["uid", "hist_score"],
        ascending=[True, False]
    )

    result = {}

    for uid, group in history.groupby("uid"):
        result[uid] = list(zip(
            group["item_id"].astype(int),
            group["hist_score"].astype(float)
        ))

    return result


hist_recommendations = generate_history_candidates(
    train_split=train_split,
    user_ids=user_ids,
    cutoff=time_cutoff
)

In [104]:
hist_sizes = pd.Series({
    uid: len(items)
    for uid, items in hist_recommendations.items()
})

print("Пользователей с историческими кандидатами:", len(hist_recommendations))
print("Среднее количество кандидатов:", hist_sizes.mean())
print("Медиана:", hist_sizes.median())
print("Максимум:", hist_sizes.max())
print("Всего пар пользователь-трек:", hist_sizes.sum())

Пользователей с историческими кандидатами: 8711
Среднее количество кандидатов: 329.3158075995867
Медиана: 371.0
Максимум: 524
Всего пар пользователь-трек: 2868670


In [105]:
# Приводим исторических кандидатов к формату uid -> set(item_id)
if "hist_candidates" in globals():
    candidates_df = hist_candidates
elif "hist_recommendations" in globals():
    candidates_df = hist_recommendations
else:
    raise NameError("Не найдена переменная с историческими кандидатами")

if isinstance(candidates_df, pd.DataFrame):
    hist_candidates_dict = (
        candidates_df.groupby("uid")["item_id"]
        .apply(set)
        .to_dict()
    )
else:
    hist_candidates_dict = {
        uid: {
            x[0] if isinstance(x, (tuple, list)) else x
            for x in items
        }
        for uid, items in candidates_df.items()
    }

# Считаем покрытие validation
user_recalls = []
total_hits = 0
total_relevant = 0

for uid, relevant_items in valid_relevant.items():
    relevant_items = set(relevant_items)
    if not relevant_items:
        continue

    candidates = hist_candidates_dict.get(uid, set())
    hits = len(candidates & relevant_items)

    user_recalls.append(hits / len(relevant_items))
    total_hits += hits
    total_relevant += len(relevant_items)

print(f"Пользователей с релевантными треками: {len(user_recalls)}")
print(f"Macro Recall: {np.mean(user_recalls):.6f}")
print(f"Micro Recall: {total_hits / total_relevant:.6f}")
print(f"Найдено релевантных пар: {total_hits:,} из {total_relevant:,}")

Пользователей с релевантными треками: 8199
Macro Recall: 0.306345
Micro Recall: 0.270618
Найдено релевантных пар: 642,454 из 2,374,022


In [108]:
def get_item_set(recommendations):
    return {
        item[0] if isinstance(item, (tuple, list)) else item
        for item in recommendations
    }

In [110]:
# Объединяем кандидатов из трёх источников
all_candidates = {}

all_uids = (
    set(hist_candidates_dict)
    | set(ials_recommendations_best)
    | set(bpr_recommendations)
)

for uid in all_uids:
    candidates = set()
    candidates.update(get_item_set(hist_candidates_dict.get(uid, set())))
    candidates.update(get_item_set(ials_recommendations_best.get(uid, [])))
    candidates.update(get_item_set(bpr_recommendations.get(uid, [])))
    all_candidates[uid] = candidates


user_recalls = []

for uid, relevant_items in valid_relevant.items():
    relevant_items = set(relevant_items)
    if relevant_items:
        hits = len(all_candidates.get(uid, set()) & relevant_items)
        user_recalls.append(hits / len(relevant_items))

print(f"Recall: {np.mean(user_recalls):.6f}")

Recall: 0.408957


In [111]:
print(ranker_data.columns.tolist())

['uid', 'item_id', 'ials_score', 'bpr_score', 'ials_rank', 'bpr_rank', 'in_ials', 'in_bpr', 'item_popularity', 'user_history_size', 'target']


In [113]:
# Исторические кандидаты
hist_df = pd.DataFrame(
    [
        (uid, item_id)
        for uid, items in hist_candidates_dict.items()
        for item_id in items
    ],
    columns=["uid", "item_id"]
)

# Прослушивания
listens = train_split.loc[
    train_split["event_type"] == "listen",
    ["uid", "item_id", "timestamp"]
].copy()

listens["age"] = (time_cutoff - listens["timestamp"]) / 86400

# Веса свежести
listens["w7"] = np.exp(-listens["age"] / 7)
listens["w30"] = np.exp(-listens["age"] / 30)

# Агрегация признаков
history_features = (
    listens.groupby(["uid", "item_id"], sort=False)
    .agg(
        ui_cnt=("timestamp", "size"),
        ui_last_age=("age", "min"),
        ui_w7=("w7", "sum"),
        ui_w30=("w30", "sum")
    )
    .reset_index()
)

# Ранги
for col in ["ui_w7", "ui_w30", "ui_cnt"]:
    history_features[f"hist_rank_{col}"] = (
        history_features.groupby("uid")[col]
        .rank(method="first", ascending=False)
    )

hist_features = hist_df.merge(
    history_features,
    on=["uid", "item_id"],
    how="left"
)

In [115]:
print(hist_features.shape)
print(hist_features.isna().sum())

(2868670, 9)
uid                 0
item_id             0
ui_cnt              0
ui_last_age         0
ui_w7               0
ui_w30              0
hist_rank_ui_w7     0
hist_rank_ui_w30    0
hist_rank_ui_cnt    0
dtype: int64


In [116]:
print(ranker_data[[
    "uid", "item_id",
    "item_popularity", "user_history_size"
]].head())

   uid  item_id  item_popularity  user_history_size
0    1   885260         9.359880           5.442418
1    1   810508         8.815222           5.442418
2    1   926224         9.087834           5.442418
3    1   181267         9.485317           5.442418
4    1   675859         6.536692           5.442418


In [117]:
# Все кандидаты из трёх источников
candidates_df = pd.DataFrame(
    [
        (uid, item_id)
        for uid, items in all_candidates.items()
        for item_id in items
    ],
    columns=["uid", "item_id"]
)

# Признаки iALS и BPR
model_features = ranker_data[
    [
        "uid", "item_id",
        "ials_score", "bpr_score",
        "ials_rank", "bpr_rank",
        "in_ials", "in_bpr"
    ]
].drop_duplicates(["uid", "item_id"])

# Признаки популярности трека
item_features = ranker_data[
    ["item_id", "item_popularity"]
].drop_duplicates("item_id")

# Признаки истории пользователя
user_features = ranker_data[
    ["uid", "user_history_size"]
].drop_duplicates("uid")

# Объединяем признаки
ranker_data_all = (
    candidates_df
    .merge(model_features, on=["uid", "item_id"], how="left")
    .merge(hist_features, on=["uid", "item_id"], how="left")
    .merge(item_features, on="item_id", how="left")
    .merge(user_features, on="uid", how="left")
)

# Признаки отсутствующих источников заполняем нулями
feature_columns = [
    "ials_score", "bpr_score",
    "ials_rank", "bpr_rank",
    "in_ials", "in_bpr",
    "ui_cnt", "ui_last_age", "ui_w7", "ui_w30",
    "hist_rank_ui_w7", "hist_rank_ui_w30", "hist_rank_ui_cnt",
    "item_popularity", "user_history_size"
]

ranker_data_all[feature_columns] = (
    ranker_data_all[feature_columns].fillna(0)
)

# Целевая переменная
positive_pairs = val_split[["uid", "item_id"]].drop_duplicates()
positive_pairs["target"] = 1

ranker_data_all = ranker_data_all.merge(
    positive_pairs,
    on=["uid", "item_id"],
    how="left"
)

ranker_data_all["target"] = (
    ranker_data_all["target"].fillna(0).astype("int8")
)

print(ranker_data_all.shape)
print(ranker_data_all["target"].sum())

(5993188, 18)
861978


In [119]:
# Сохраняем все положительные примеры
positive_train = train_data_all[
    train_data_all["target"] == 1
]

# Оставляем до 100 отрицательных примеров на пользователя
negative_train = train_data_all[
    train_data_all["target"] == 0
].copy()

rng = np.random.default_rng(42)
negative_train["_random"] = rng.random(len(negative_train))

negative_train = (
    negative_train
    .sort_values(["uid", "_random"])
    .groupby("uid", sort=False)
    .head(100)
    .drop(columns="_random")
)

# Формируем обучающую выборку
train_data_sampled = (
    pd.concat([positive_train, negative_train])
    .sort_values("uid")
)

# Создаём Pool
train_pool_sampled = Pool(
    train_data_sampled[feature_columns],
    label=train_data_sampled["target"],
    group_id=train_data_sampled["uid"]
)

# Валидацию оставляем без изменений
test_pool_all = Pool(
    test_data_all[feature_columns],
    label=test_data_all["target"],
    group_id=test_data_all["uid"]
)

# Обучаем CatBoost
ranker_all = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=300,
    learning_rate=0.08,
    depth=6,
    random_seed=42,
    thread_count=4,
    verbose=50,
    early_stopping_rounds=30
)

ranker_all.fit(
    train_pool_sampled,
    eval_set=test_pool_all,
    use_best_model=True
)

0:	test: 0.2836492	best: 0.2836492 (0)	total: 1.15s	remaining: 5m 43s
50:	test: 0.6117059	best: 0.6119476 (47)	total: 58s	remaining: 4m 43s
100:	test: 0.6204845	best: 0.6206501 (91)	total: 1m 54s	remaining: 3m 45s
150:	test: 0.6237343	best: 0.6243577 (148)	total: 2m 49s	remaining: 2m 47s
200:	test: 0.6267075	best: 0.6267075 (200)	total: 3m 44s	remaining: 1m 50s
250:	test: 0.6283496	best: 0.6283496 (250)	total: 4m 40s	remaining: 54.7s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 0.628595527
bestIteration = 264

Shrink model to first 265 iterations.


CatBoostRanker(depth=6, early_stopping_rounds=30, eval_metric='NDCG:top=10', iterations=300, learning_rate=0.08, loss_function='YetiRank', random_seed=42, thread_count=4, verbose=50)

In [120]:
# Оставляем пользователей, у которых есть хотя бы один положительный кандидат
user_targets_all = ranker_data_all.groupby("uid")["target"].sum()
users_final = user_targets_all[user_targets_all > 0].index

train_data_final = (
    ranker_data_all[
        ranker_data_all["uid"].isin(users_final)
    ]
    .sort_values("uid")
)

train_pool_final = Pool(
    train_data_final[feature_columns],
    label=train_data_final["target"],
    group_id=train_data_final["uid"]
)

# Финальная модель
ranker_final = CatBoostRanker(
    loss_function="YetiRank",
    iterations=265,
    learning_rate=0.08,
    depth=6,
    random_seed=42,
    thread_count=4,
    verbose=50
)

ranker_final.fit(train_pool_final)

0:	total: 8.57s	remaining: 37m 42s


KeyboardInterrupt: 

In [121]:
for name in [
    "ials_model",
    "model",
    "bpr_model",
    "user_ids",
    "item_ids",
    "R"
]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj), getattr(obj, "shape", ""))

ials_model <class 'implicit.cpu.als.AlternatingLeastSquares'> 
model <class 'implicit.cpu.als.AlternatingLeastSquares'> 
bpr_model <class 'implicit.cpu.bpr.BayesianPersonalizedRanking'> 
user_ids <class 'pandas.Index'> (8711,)
item_ids <class 'pandas.Index'> (781188,)
R <class 'scipy.sparse._csr.csr_matrix'> (8711, 781188)


In [124]:
test_uid_set = set(test_users["user_id"])
known_uid_set = set(user_ids)

print("Тестовых пользователей:", len(test_uid_set))
print("Есть в iALS/BPR:", len(test_uid_set & known_uid_set))
print("Нет в iALS/BPR:", len(test_uid_set - known_uid_set))

Тестовых пользователей: 9213
Есть в iALS/BPR: 8687
Нет в iALS/BPR: 526


In [123]:
print(test_users.head())
print(test_users.columns)

   user_id
0        1
1        3
2        4
3        5
4        6
Index(['user_id'], dtype='str')


In [125]:
print("ials_model:", ials_model.factors, ials_model.alpha)
print("model:", model.factors, model.alpha)
print("bpr_model:", bpr_model.factors)

ials_model: 64 20
model: 128 10
bpr_model: 128


In [126]:
print("train:", train.shape)
print("train_split:", train_split.shape)
print("R:", R.shape)
print("R_binary:", R_binary.shape)

train: (43000347, 7)
train_split: (38701795, 7)
R: (8711, 781188)
R_binary: (8711, 781188)


In [127]:
R_full, user_ids_full, item_ids_full = build_interaction_matrix(train)

R_binary_full = R_full.copy()
R_binary_full.data[:] = 1

user_to_idx_full = {
    uid: idx for idx, uid in enumerate(user_ids_full)
}

print("Матрица:", R_full.shape)

Матрица: (9086, 827802)


In [ ]:
# iALS — лучшие найденные параметры
ials_model_full = fit_ials(
    R_full,
    factors=128,
    regularization=0.01,
    iterations=20,
    alpha=10
)

# BPR — ранее проверенные параметры
bpr_model_full = BayesianPersonalizedRanking(
    factors=128,
    learning_rate=0.05,
    regularization=0.01,
    iterations=30,
    num_threads=4,
    random_state=42
)

bpr_model_full.fit(R_binary_full)

 20%|████████████████▌                                                                  | 4/20 [01:04<04:18, 16.14s/it]

In [ ]:
known_test_users = [
    uid for uid in test_users["user_id"]
    if uid in user_to_idx_full
]

test_user_indices = np.array([
    user_to_idx_full[uid] for uid in known_test_users
])

test_user_items = R_full[test_user_indices]

# iALS
ials_items, ials_scores = ials_model_full.recommend(
    userid=test_user_indices,
    user_items=test_user_items,
    N=300,
    filter_already_liked_items=False
)

ials_recommendations_test = {
    uid: [
        (int(item_ids_full[idx]), float(score))
        for idx, score in zip(ials_items[i], ials_scores[i])
    ]
    for i, uid in enumerate(known_test_users)
}

# BPR
bpr_items, bpr_scores = bpr_model_full.recommend(
    userid=test_user_indices,
    user_items=R_binary_full[test_user_indices],
    N=300,
    filter_already_liked_items=False
)

bpr_recommendations_test = {
    uid: [
        (int(item_ids_full[idx]), float(score))
        for idx, score in zip(bpr_items[i], bpr_scores[i])
    ]
    for i, uid in enumerate(known_test_users)
}